<a href="https://colab.research.google.com/github/t6niskoppel/Optimization-for-Robot-Motion-Planning-and-Control-assignment3/blob/main/assignment_3_tonis_koppel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 3 — MPC Navigation in Cluttered Environments

Differential-drive robot navigating from LiDAR perception, using trajectory
optimization at every control loop as **Model Predictive Control** feedback.

Four planners (selected via `planner_type`), all implemented in `planner.py`:

- **`gd`** — plain gradient descent
- **`nesterov`** — Nesterov accelerated gradient
- **`cem`** — Cross-Entropy Method
- **`hybrid`** — CEM followed by Adam refinement

## Setup

In [ ]:
!pip install ir-sim[all]


In [ ]:
# Setup. On Colab: clone the repo (or pull if a reused runtime already has it).
# Locally: do NOT clone -- just use this working checkout as-is, so your local
# edits are what runs. Pure os.chdir (no %cd magic, which errors off-Colab).
import os, subprocess

REPO = "https://github.com/t6niskoppel/Optimization-for-Robot-Motion-Planning-and-Control-assignment3.git"
if os.path.isdir("/content"):                         # Google Colab
    os.chdir("/content")
    if os.path.isdir("opt_irsim/.git"):               # reused runtime -> pull latest
        subprocess.run(["git", "-C", "opt_irsim", "pull", "-q"])
    else:
        subprocess.run(["git", "clone", "-q", REPO, "opt_irsim"])
    os.chdir("opt_irsim")
elif os.path.isdir("opt_irsim") and os.path.basename(os.getcwd()) != "opt_irsim":
    os.chdir("opt_irsim")                              # local, launched from the parent dir
print("working dir:", os.getcwd())

## 1. Visualize each planner on one world

`benchmark(world, save_ani=True)` runs every planner on a single world and saves
one animation per planner into `animation/<world>/<planner>.gif`, so all the
planners for that world sit in the same folder for side-by-side comparison.

In [ ]:
import sys, os

# Put cwd first on sys.path so `from test import ...` finds the local opt_irsim/test.py
# and not Python's stdlib `test` package. Also drop any already-imported stdlib `test`
# from a prior run in this kernel, so re-running the cell can't keep the wrong module.
sys.path.insert(0, os.getcwd())
sys.modules.pop("test", None)

from test import benchmark, barn_worlds

planners = ["gd", "nesterov", "cem", "hybrid"]
vis_world = "barn_envs/barn_43.yaml"

# One world, every planner, saving a gif per planner into animation/<world>/.
single = benchmark(vis_world, planners=planners, save_ani=True)

import os
from IPython.display import Image, display

# Show the gifs saved above: animation/<world>/<planner>.gif, one per planner.
stem = os.path.splitext(os.path.basename(vis_world))[0]
for pt in planners:
    g = f"animation/{stem}/{pt}.gif"
    if os.path.exists(g):
        print(f"{stem} — {pt}")
        display(Image(filename=g))

### Saved animations

In [ ]:
import os
from IPython.display import Image, display

# Show the gifs saved above: animation/<world>/<planner>.gif, one per planner.
stem = os.path.splitext(os.path.basename(vis_world))[0]
for pt in planners:
    g = f"animation/{stem}/{pt}.gif"
    if os.path.exists(g):
        print(f"{stem} — {pt}")
        display(Image(filename=g))

## 2. Benchmark across BARN environments

`benchmark` takes a single world or a list of worlds and runs each world on
every planner (world-outer, planner-inner). `save_ani=False` skips all plotting
for a fast metrics run; `save_ani=True` also saves a gif per (planner, world).

Reported per planner: **success rate** (reached goal *and* no collision),
**collision rate**, **time-to-reach-goal**, **min clearance** to obstacles,
and mean **solve time per MPC step**.

In [ ]:
# Fast metrics-only run (no gifs for the passes).
#
# barn_worlds accepts either:
#   * an int  -> the first n worlds, e.g. barn_worlds(10)
#   * a list/range of indices -> exactly those worlds, in the order given:
#       barn_worlds(range(0, 300, 10))   # every 10th world: barn_0, barn_10, ...
#       barn_worlds([3, 43, 100])        # specific worlds
# barn_worlds(300) is every barn world. The LiDAR downsample runs every MPC step,
# so this is the slow part of the notebook -- shrink the selection for a quick check.
#
# The benchmark is crash-resilient: if one world/planner throws, it is logged as a
# failure and the sweep keeps going; envs and figures are freed between episodes.
#
# save_failures=True keeps the fast metrics sweep, but re-runs only the failing
# (world, planner) episodes with animation and writes them to
# failures/<world>__<planner>.gif so each failure can be inspected. Runs are
# deterministic, so the re-run reproduces the exact failure.
#
# Slow variant that also saves a gif per planner per world (writes many gifs):
#     benchmark(barn_worlds([3, 43]), planners=planners, save_ani=True)

# World selection: every world 0-49, then every 2nd from 50 to 150, then every 5th
# from 150 to 300 -- dense over the low indices, thinning out the tail to bound run
# time. (130 worlds total.)
sweep_worlds = list(range(0, 50)) + list(range(50, 150, 2)) + list(range(150, 300, 5))
results = benchmark(barn_worlds(sweep_worlds), planners=planners, save_ani=False, save_failures=True)

### Metrics table

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
summary = df.groupby('planner').agg(
    success_rate=('success', 'mean'),
    collision_rate=('collided', 'mean'),
    mean_time_to_goal=('time_to_goal', 'mean'),
    mean_min_clearance=('min_clearance', 'mean'),
    mean_solve_ms=('solve_ms', 'mean'),
    n=('success', 'count'),
)
summary['success_rate'] = (summary['success_rate'] * 100).round(0)
summary['collision_rate'] = (summary['collision_rate'] * 100).round(0)
summary = summary.round(2)
summary

### Metrics graph

Success rate, collision rate, and mean time-to-goal per planner across the
benchmarked worlds.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image

# bar charts of the headline metrics per planner. test.py forces the Agg backend,
# so save to png and display it rather than relying on inline rendering.
order = ["gd", "nesterov", "cem", "hybrid"]
present = [p for p in order if p in df["planner"].unique()]
g = df.groupby("planner")

panels = [
    ("success", lambda s: s.mean() * 100, "Success rate (%)", "tab:green", (0, 100)),
    ("collided", lambda s: s.mean() * 100, "Collision rate (%)", "tab:red", (0, 100)),
    ("time_to_goal", lambda s: s.mean(), "Mean time-to-goal (s)", "tab:blue", None),
]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (col, fn, title, color, ylim) in zip(axes, panels):
    vals = fn(g[col]).reindex(present)
    ax.bar(present, vals.values, color=color)
    ax.set_title(title)
    if ylim:
        ax.set_ylim(*ylim)
    ax.grid(axis="y", alpha=0.3)
    for i, v in enumerate(vals.values):
        if v == v:  # skip NaN (planner never reached the goal)
            ax.text(i, v, f"{v:.1f}", ha="center", va="bottom", fontsize=9)

fig.suptitle(f"Planner comparison across {df['world'].nunique()} barn worlds")
fig.tight_layout()
fig.savefig("metrics.png", dpi=120, bbox_inches="tight")
Image("metrics.png")